# Aileron Design — As-Selected Re-Solve (COTS Servo)
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

NB4 (`aileron_design`) sized the ailerons against a *configured estimate*
of the servo (`config/aileron.yaml`: 12 g, 9g-class). NB11
(`cots_selection`) has since frozen a real part. This notebook **re-solves
the aileron design with the frozen servo** — as a *new, downstream*
notebook, so the design pipeline stays one-way (ADR-0001/ADR-0012): the
NB4 solution remains the record of what the estimate gave, this one is the
as-selected solution, and the delta between them is a finding against the
estimate.

What actually changes: the aileron *surface* sizing (geometry, flap
effectiveness, roll authority) does not depend on the servo at all, so it
is re-derived identically; the **hardware mass** (carved from the mass
fractions by the fuselage re-solve, NB14) and the **torque check** — now
against a real stall-torque rating instead of a class assumption — do.

## Inputs

- `out/components.yaml` *(NB11 — the frozen servo)*
- `out/airfoil.yaml`, `out/control_vanes.yaml` *(NB2/NB3, unchanged)*
- `out/aileron.yaml` *(NB4 — the conceptual solution, for the delta report)*
- `config/aileron.yaml`, `config/aerodynamics.yaml`

## Outputs

- `out/aileron_cots.yaml` *(same schema as `out/aileron.yaml` + frozen-servo
  block; consumed by NB14)*

---

In [1]:
import sys, math
from pathlib import Path
from dataclasses import replace
import yaml

REPO_ROOT   = Path().resolve().parents[0]
SRC_PATH    = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config"
OUT_PATH    = REPO_ROOT / "out"
sys.path.insert(0, str(SRC_PATH))


# Section 1 — Design Inputs

Re-run the sizing loop from `config/` — same pattern as NB2–NB11, so this
notebook stays consistent with the upstream design state — and load the
wing/vane handoffs plus the frozen COTS hardware list.

---

In [2]:
from conceptual_design import (
    run_sizing_loop, FrozenComponents,
    Environment, Mission, Aerodynamics, Battery,
    WeightFraction, PropulsiveSystemParameters,
)
from conceptual_design.forward_flight_power import ForwardFlightParams
from conceptual_design.wing_sizing import WingStructureParams
from conceptual_design.models import RotorParams, Avionics
from conceptual_design.aileron_design import AileronParams, size_aileron, write_aileron_yaml

env     = Environment()
mission = Mission.from_yaml(CONFIG_PATH / "mission.yaml")
aero    = Aerodynamics.from_yaml(CONFIG_PATH / "aerodynamics.yaml")
batt    = Battery.from_yaml(CONFIG_PATH / "battery.yaml")
wf      = WeightFraction.from_yaml(CONFIG_PATH / "initial_weight_fraction_estimation.yaml")
prop    = PropulsiveSystemParameters.from_yaml(CONFIG_PATH / "propulsive_system_parameters.yaml")
ff      = ForwardFlightParams.from_yaml(CONFIG_PATH / "forward_flight_params.yaml")
ws      = WingStructureParams.from_yaml(CONFIG_PATH / "wing_structure_params.yaml")
rotor    = RotorParams.from_yaml(CONFIG_PATH / "rotor.yaml")
avionics = Avionics.from_yaml(CONFIG_PATH / "avionics.yaml")
ail_p    = AileronParams.from_yaml(CONFIG_PATH / "aileron.yaml")

result = run_sizing_loop(
    m_payload_kg=mission.payload_kg, mission=mission, aero=aero, batt=batt,
    wf=wf, prop_params=prop, ff_params=ff, ws_params=ws, env=env,
    D_rotor_m=rotor.D_rotor_m, P_hotel_W=avionics.P_hotel_W,
)

af      = yaml.safe_load((OUT_PATH / "airfoil.yaml").read_text(encoding="utf-8"))
vanes   = yaml.safe_load((OUT_PATH / "control_vanes.yaml").read_text(encoding="utf-8"))
ail_est = yaml.safe_load((OUT_PATH / "aileron.yaml").read_text(encoding="utf-8"))
comps   = FrozenComponents.from_yaml(OUT_PATH / "components.yaml")
servo   = comps["servo"]

MTOW_kg      = result.m_total_kg
W_N          = MTOW_kg * env.g
ddot_min     = aero.ddot_min_deg_s2

print(f"Converged MTOW : {MTOW_kg:.3f} kg  ({W_N:.2f} N)")
print(f"Wing            : b = {result.wing.b_wing*1e3:.0f} mm, MAC = {result.wing.chord_mean*1e3:.1f} mm")
print(f"Frozen servo    : {servo.name}  ({servo.id})")
print(f"  mass           : {servo.mass_kg*1e3:.1f} g   (NB4 estimate: {ail_p.servo_mass_kg*1e3:.1f} g)")
print(f"  stall torque   : {servo.ratings['stall_torque_gcm']:.0f} g cm")
print(f"ddot_min        : {ddot_min:.0f} deg/s^2  (config/aerodynamics.yaml, shared with NB3/NB4)")


Converged MTOW : 2.303 kg  (22.58 N)
Wing            : b = 1049 mm, MAC = 174.8 mm
Frozen servo    : KST X08 V6 (metal gear, digital)  (kst_x08_v6)
  mass           : 8.9 g   (NB4 estimate: 12.0 g)
  stall torque   : 1400 g cm
ddot_min        : 30 deg/s^2  (config/aerodynamics.yaml, shared with NB3/NB4)


# Section 2 — Re-Solve with the Frozen Servo

Same physics as NB4 (`size_aileron`, one call), with the configured servo
mass estimate replaced by the frozen part's catalog mass. The residual
jet-vane roll authority in cruise is re-derived with the same
thrust-scaling law as NB4. The torque check is now a **real margin**:
catalog stall torque against the hinge-moment requirements of *both*
surfaces the single servo type drives (jet vanes, NB3; ailerons, this
notebook).

---

In [3]:
ail_p_cots = replace(ail_p, servo_mass_kg=servo.mass_kg)

ail = size_aileron(
    b_wing_m=result.wing.b_wing, chord_mean_m=result.wing.chord_mean,
    tc_ratio=af["tc_ratio"], AR=aero.AR, e_oswald=af["e_oswald"],
    V_cruise=mission.V_cruise, rho=env.rho,
    I_roll=vanes["I_roll_kgm2"], p=ail_p_cots,
)

# residual jet-vane authority in cruise (same law as NB4)
T_hover  = prop.s_ratio * W_N
T_cruise = W_N / af["LD_cruise"]
thrust_ratio = T_cruise / T_hover
ddot_roll_vane_cruise = vanes["ddot_roll_deg_s2"] * thrust_ratio
ddot_roll_total = ail.ddot_roll_deg_s2 + ddot_roll_vane_cruise
cruise_ok = ddot_roll_total >= ddot_min

stall_gcm   = float(servo.ratings["stall_torque_gcm"])
margin_ail  = stall_gcm / ail.servo_torque_req_gcm
margin_vane = stall_gcm / vanes["servo_torque_req_gcm"]

print(f"Aileron chord / span  : {ail.c_aileron_m*1e3:.1f} / {ail.b_aileron_m*1e3:.1f} mm  (geometry unchanged from NB4)")
print(f"Combined ddot_roll    : {ddot_roll_total:.1f} deg/s^2  (req {ddot_min:.0f})  -> {'OK' if cruise_ok else 'FAIL'}")
print()
print(f"{'surface':<12}{'torque req [g cm]':>19}{'stall [g cm]':>14}{'margin':>9}")
print("-" * 54)
print(f"{'aileron':<12}{ail.servo_torque_req_gcm:>19.1f}{stall_gcm:>14.0f}{margin_ail:>8.1f}x")
print(f"{'jet vane':<12}{vanes['servo_torque_req_gcm']:>19.1f}{stall_gcm:>14.0f}{margin_vane:>8.1f}x")

# the frozen servo must actually drive both surfaces it was selected for
assert cruise_ok, "cruise roll authority lost in the re-solve -- upstream drift"
assert margin_ail >= 1.0 and margin_vane >= 1.0, (
    "frozen servo stall torque below a hinge-moment requirement -- "
    "the design has outgrown the frozen hardware")


Aileron chord / span  : 21.0 / 62.9 mm  (geometry unchanged from NB4)
Combined ddot_roll    : 560.2 deg/s^2  (req 30)  -> OK

surface       torque req [g cm]  stall [g cm]   margin
------------------------------------------------------
aileron                    75.7          1400    18.5x
jet vane                  441.5          1400     3.2x


# Section 3 — Output Export

`out/aileron_cots.yaml` — same schema as `out/aileron.yaml` (so the
fuselage re-solve consumes it identically) plus a `cots_servo` provenance
block.

---

In [4]:
write_aileron_yaml(
    ail, ail_p_cots, OUT_PATH / "aileron_cots.yaml",
    ddot_roll_vane_cruise_deg_s2=ddot_roll_vane_cruise,
    ddot_min_deg_s2=ddot_min,
    regen_notebook="notebooks/aileron_design_cots.ipynb",
    extra={"cots_servo": {
        "id":             servo.id,
        "name":           servo.name,
        "mass_g":         round(servo.mass_kg * 1e3, 2),
        "stall_torque_gcm": stall_gcm,
        "margin_vs_aileron_req": round(margin_ail, 2),
        "margin_vs_vane_req":    round(margin_vane, 2),
    }},
)
print(f"As-selected aileron design written -> {OUT_PATH / 'aileron_cots.yaml'}")


As-selected aileron design written -> D:\Dev\vbat-uav-notebooks\out\aileron_cots.yaml


# Section 4 — Delta vs. the Conceptual Solution (NB4) and Summary

The delta table is the point of the re-solve: what did freezing real
hardware change against the estimate the conceptual design used?

---

In [5]:
hw_est  = 2 * (ail_est["servo_mass_kg_each"] + ail_est["linkage_mass_kg_each"]) * 1e3
hw_cots = 2 * (servo.mass_kg + ail_p.linkage_mass_kg) * 1e3

bar = "=" * 62
print(bar)
print("  AILERON AS-SELECTED RE-SOLVE SUMMARY".center(62))
print(bar)
print(f"  {'quantity':<30}{'NB4 (est.)':>12}{'this NB':>12}")
print("  " + "-" * 58)
print(f"  {'servo mass, each [g]':<30}{ail_est['servo_mass_kg_each']*1e3:>12.1f}{servo.mass_kg*1e3:>12.1f}")
print(f"  {'aileron hw total (2x) [g]':<30}{hw_est:>12.1f}{hw_cots:>12.1f}")
print(f"  {'servo torque req [g cm]':<30}{ail_est['servo_torque_req_gcm']:>12.1f}{ail.servo_torque_req_gcm:>12.1f}")
print(f"  {'combined cruise ddot [d/s2]':<30}{ail_est['ddot_roll_total_cruise_deg_s2']:>12.1f}{ddot_roll_total:>12.1f}")
print()
print(f"  {'frozen servo':<30}: {servo.id}  ({servo.mass_kg*1e3:.1f} g, {stall_gcm:.0f} g cm)")
print(f"  {'stall margin vane / aileron':<30}: {margin_vane:.1f}x / {margin_ail:.1f}x")
print(f"  {'hardware delta vs NB4':<30}: {hw_cots - hw_est:+.1f} g  (into NB14 CG re-solve)")
print(bar)


              AILERON AS-SELECTED RE-SOLVE SUMMARY            
  quantity                        NB4 (est.)     this NB
  ----------------------------------------------------------
  servo mass, each [g]                  12.0         8.9
  aileron hw total (2x) [g]             30.0        23.8
  servo torque req [g cm]               75.7        75.7
  combined cruise ddot [d/s2]          560.2       560.2

  frozen servo                  : kst_x08_v6  (8.9 g, 1400 g cm)
  stall margin vane / aileron   : 3.2x / 18.5x
  hardware delta vs NB4         : -6.2 g  (into NB14 CG re-solve)
